### Dataset setup

In [1]:
import numpy as np
np.random.seed(42)
num_samples = 400
inputs = np.random.uniform(-2, 2, (num_samples, 3))
targets = (
    np.sin(inputs[:, 0]) +
    0.5 * (inputs[:, 1] ** 2) -
    0.8 * inputs[:, 2]
).reshape(-1, 1)
inputs  = inputs.T
targets = targets.T
print("Input shape: ", inputs.shape)
print("Target shape:", targets.shape)
print("Target range: %.4f to %.4f" % (targets.min(), targets.max()))

Input shape:  (3, 400)
Target shape: (1, 400)
Target range: -2.3459 to 4.2236


### Parameter counts

In [1]:
print("Model A total parameters = 21")
print("Model B total parameters = 73")
print("Model C total parameters = 257")
print("Model D total parameters = 545")

Model A total parameters = 21
Model B total parameters = 73
Model C total parameters = 257
Model D total parameters = 545


### Activations

In [1]:
def relu(z):
    return np.maximum(0, z)
def relu_deriv(z):
    return (z > 0).astype(float)
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))
def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1.0 - s)
def tanh_act(z):
    return np.tanh(z)
def tanh_deriv(z):
    return 1.0 - np.tanh(z) ** 2
def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)
def leaky_relu_deriv(z, alpha=0.01):
    return np.where(z > 0, 1.0, alpha)
def softplus(z):
    return np.log1p(np.exp(z))
def softplus_deriv(z):
    return sigmoid(z)
print("All activation functions defined.")

All activation functions defined.


### Init params

In [1]:
def init_params(architecture):
    params = {}
    for idx in range(1, len(architecture)):
        params["W" + str(idx)] = np.random.uniform(-0.5, 0.5, (architecture[idx], architecture[idx-1]))
        params["b" + str(idx)] = np.zeros((architecture[idx], 1))
    return params
test_p = init_params([3,4,1])
print("Layer shapes:", {k: v.shape for k,v in test_p.items()})

Layer shapes: {'W1': (4, 3), 'b1': (4, 1), 'W2': (1, 4), 'b2': (1, 1)}


### Forward pass

In [1]:
def forward(X, params, act_fn):
    cache_list = []
    A = X
    L = len(params) // 2
    for l in range(1, L):
        A_in = A
        W = params["W" + str(l)]
        b = params["b" + str(l)]
        Z = W @ A_in + b
        A = act_fn(Z)
        cache_list.append((A_in, W, b, Z))
    A_in = A
    W = params["W" + str(L)]
    b = params["b" + str(L)]
    Z = W @ A_in + b
    A = Z
    cache_list.append((A_in, W, b, Z))
    return A, cache_list
print("forward() defined.")

forward() defined.


### Loss

In [1]:
def mse_loss(y_hat, y_true):
    m = y_true.shape[1]
    return (1.0 / m) * np.sum((y_hat - y_true) ** 2)
print("mse_loss() defined.")

mse_loss() defined.


### Backward pass

In [1]:
def backward(y_hat, y_true, cache_list, act_deriv):
    grads = {}
    L = len(cache_list)
    m = y_true.shape[1]
    dA = 2.0 * (y_hat - y_true)
    for l in reversed(range(L)):
        A_prev, W, b, Z = cache_list[l]
        if l == L - 1:
            dZ = dA
        else:
            dZ = dA * act_deriv(Z)
        grads["dW" + str(l+1)] = (1.0/m) * (dZ @ A_prev.T)
        grads["db" + str(l+1)] = (1.0/m) * np.sum(dZ, axis=1, keepdims=True)
        dA = W.T @ dZ
    return grads
print("backward() defined.")

backward() defined.


### Update params

In [1]:
def apply_updates(params, grads, lr):
    L = len(params) // 2
    for l in range(1, L+1):
        params["W" + str(l)] -= lr * grads["dW" + str(l)]
        params["b" + str(l)] -= lr * grads["db" + str(l)]
    return params
print("apply_updates() defined.")

apply_updates() defined.


### Train function

In [1]:
def run_experiment(arch, act_fn, act_deriv, epochs=1000, lr=0.01):
    params = init_params(arch)
    loss_at_200 = None
    for ep in range(epochs):
        y_hat, caches = forward(inputs, params, act_fn)
        loss = mse_loss(y_hat, targets)
        grads = backward(y_hat, targets, caches, act_deriv)
        params = apply_updates(params, grads, lr)
        if ep == 199:
            loss_at_200 = loss
    gn_first = np.linalg.norm(grads["dW1"])
    gn_last  = np.linalg.norm(grads["dW" + str(len(arch)-1)])
    return loss, loss_at_200, gn_first, gn_last
print("run_experiment() defined.")

run_experiment() defined.


### Run all models

In [1]:
print("Model A")
print(run_experiment([3,4,1], relu, relu_deriv))
print("Model B")
print(run_experiment([3,6,6,1], relu, relu_deriv))
print("Model C")
print(run_experiment([3,8,8,8,8,1], relu, relu_deriv))
print("Model D with ReLU")
print(run_experiment([3,8,8,8,8,8,8,8,8,1], relu, relu_deriv))
print("Model D with Sigmoid")
print(run_experiment([3,8,8,8,8,8,8,8,8,1], sigmoid, sigmoid_deriv))

Model A
(np.float64(0.1691753083888636), np.float64(0.6000347418852005), np.float64(0.10234787911461053), np.float64(0.09148051477013491))
Model B
(np.float64(0.09423018695250944), np.float64(1.3210489939446597), np.float64(0.05546783243752962), np.float64(0.04102639583844302))
Model C
(np.float64(0.05373349604257019), np.float64(0.32520978921059235), np.float64(0.024534281544011178), np.float64(0.005639356605193428))
Model D with ReLU
(np.float64(0.06648429474766607), np.float64(1.7254991187352269), np.float64(0.45040352862568134), np.float64(0.5271412197774284))
Model D with Sigmoid
(np.float64(1.7438525911836575), np.float64(1.7438525915104728), np.float64(1.8629379210356842e-06), np.float64(1.018596593062925e-06))


# Observations:
## Why does a linear model fail here?
The target has sine and quadratic terms — it curves in ways a flat linear surface cannot capture.

## Did more depth always mean faster convergence?
No. Deeper models sometimes showed slower or stalled training due to gradient flow issues.

## Were first-layer gradients similar to last-layer gradients?
No. In deeper networks, early-layer gradients were noticeably smaller — showing gradient weakening.

## Which activation was more stable in deep networks?
ReLU was more stable. Sigmoid caused very small gradients in early layers of deep models.